In [1]:
%load_ext dotenv
%dotenv

In [2]:
import pandas as pd
import numpy as np
import pprint
from sentence_transformers import SentenceTransformer
import os
from pinecone import Pinecone, ServerlessSpec

In [3]:
df = pd.read_csv("data/alldata_1_for_kaggle.csv", encoding='latin1')

# Fixing oddly named columns of '0' and 'a'
df = df.rename(columns={"0": "diagnosis", "a": "clinical_text"})

# Drop empty rows to prevent embedding errors
df = df.dropna()

# For testing purposes, let's just use the first 500 rows so it doesn't take hours to embed!
# df = df.head(500).copy()

files["metadata"] = files.apply(lambda row: {
    "course_name": row["course_name"],
    "section_name": row["section_name"],
    "section_description": row["section_description"],
}, axis = 1)

In [4]:
# Model fine-tuned on PubMed literature
model = SentenceTransformer('finetuned-pubmedbert')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:
# Specific weights for different text components
weight_diagnosis = 5
weight_abstract = 3
weight_full_text = 1 

# Pinecone metadata to return with search results
df['metadata'] = df.apply(lambda row: {
    "diagnosis": row['diagnosis'],
    # We save a 300-character snippet to display in the search results
    "text_snippet": str(row['clinical_text'])[:300] + "..." 
}, axis=1)

def create_medical_embeddings(row):
    text = str(row['clinical_text'])

    # 1. Diagnosis embedding
    emb_diagnosis = model.encode(row['diagnosis'], show_progress_bar=False) * weight_diagnosis

    # 2. "Abstract" embedding (First 200 chars usually contain the core thesis)
    abstract = text[:200]
    emb_abstract = model.encode(abstract, show_progress_bar=False) * weight_abstract

    # 3. Full Text embedding (1000 chars cap to avoid model token limits)
    full_text = text[:1000]
    emb_full_text = model.encode(full_text, show_progress_bar=False) * weight_full_text

    # Combine by averaging them based on the weights
    total_weight = weight_diagnosis + weight_abstract + weight_full_text
    combined_embedding = (emb_diagnosis + emb_abstract + emb_full_text) / total_weight

    return combined_embedding

print("Generating weighted embeddings takes a few minutes")
df['embedding'] = df.apply(create_medical_embeddings, axis=1)
df['unique_id'] = df.index.astype(str)
print("Embeddings generated!")

Generating weighted embeddings takes a few minutes
Embeddings generated!


In [13]:
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

index_name = "medical-weighted-search"

# IMPORTANT: PubMedBERT outputs 768-dimensional vectors, not 384 like MiniLM
dimension = 768 
metric = "cosine"

# Delete index if it already exists to start fresh
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} successfully deleted.")

# Create the new index
pc.create_index(
    name=index_name, 
    dimension=dimension, 
    metric=metric, 
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

index = pc.Index(index_name)

# Prepare vectors and upsert in batches                                                                                                                                                                                                                                                                                                                                                                                         
vectors_to_upsert = [
    (row['unique_id'], row['embedding'].tolist(), row['metadata'])
    for _, row in df.iterrows()                                                                                                                                                                                                                                                                                                                                                                                                 
]                                                                                                                                                                                                                                                                                                                                                                                                                               
batch_size = 100
total_batches = (len(vectors_to_upsert) + batch_size - 1) // batch_size
                                                                                                                                                                                                                                                                                                                                                                                                                                  
for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i : i + batch_size]
    index.upsert(vectors=batch)
    print(f"Upserted batch {i // batch_size + 1}/{total_batches}")

print(f"\nData successfully upserted to Pinecone index! ({len(vectors_to_upsert)} vectors total)") 

Upserted batch 1/76
Upserted batch 2/76
Upserted batch 3/76
Upserted batch 4/76
Upserted batch 5/76
Upserted batch 6/76
Upserted batch 7/76
Upserted batch 8/76
Upserted batch 9/76
Upserted batch 10/76
Upserted batch 11/76
Upserted batch 12/76
Upserted batch 13/76
Upserted batch 14/76
Upserted batch 15/76
Upserted batch 16/76
Upserted batch 17/76
Upserted batch 18/76
Upserted batch 19/76
Upserted batch 20/76
Upserted batch 21/76
Upserted batch 22/76
Upserted batch 23/76
Upserted batch 24/76
Upserted batch 25/76
Upserted batch 26/76
Upserted batch 27/76
Upserted batch 28/76
Upserted batch 29/76
Upserted batch 30/76
Upserted batch 31/76
Upserted batch 32/76
Upserted batch 33/76
Upserted batch 34/76
Upserted batch 35/76
Upserted batch 36/76
Upserted batch 37/76
Upserted batch 38/76
Upserted batch 39/76
Upserted batch 40/76
Upserted batch 41/76
Upserted batch 42/76
Upserted batch 43/76
Upserted batch 44/76
Upserted batch 45/76
Upserted batch 46/76
Upserted batch 47/76
Upserted batch 48/76
U

In [14]:
from pinecone_text.sparse import BM25Encoder                                                                                                                                                                                                                                                                                                                                                                                    
                                                                                                                                                                                                                                                                                                                                                                                                                                  
print("Fitting BM25 on corpus...")                                                                                                                                                                                                                                                                                                                                                                                              
bm25 = BM25Encoder()                                                                                                                                                                                                                                                                                                                                                                                                            
bm25.fit(df['clinical_text'].astype(str).tolist())                                                                                                                                                                                                                                                                                                                                                                              
bm25.dump("bm25_model.json")                                                                                                                                                                                                                                                                                                                                                                                                    
print("BM25 fitted and saved.")

Fitting BM25 on corpus...


  0%|          | 0/7570 [00:00<?, ?it/s]

BM25 fitted and saved.


In [15]:
new_index_name = "medical-hybrid-search"                                                                                                                                                                                                                                                                                                                                                                                        
                                                                                                                                                                                                                                                                                                                                                                                                                                  
# Delete old index                                                                                                                                                                                                                                                                                                                                                                                                              
if "medical-weighted-search" in [idx.name for idx in pc.list_indexes()]:
    pc.delete_index("medical-weighted-search")                          
    print("Deleted old cosine index")                                                                                                                                                                                                                                                                                                                                                                                           
                                     
# Delete new if already exists (clean slate)                                                                                                                                                                                                                                                                                                                                                                                    
if new_index_name in [idx.name for idx in pc.list_indexes()]:                                                                                                                                                                                                                                                                                                                                                                   
    pc.delete_index(new_index_name)                                                                                                                                                                                                                                                                                                                                                                                             
                                                                                                                                                                                                                                                                                                                                                                                                                                
pc.create_index(
    name=new_index_name,                                                                                                                                                                                                                                                                                                                                                                                                        
    dimension=768,      
    metric="dotproduct",      # Required for hybrid scaling to work
    spec=ServerlessSpec(cloud="aws", region="us-east-1")           
)                                                                                                                                                                                                                                                                                                                                                                                                                               
hybrid_index = pc.Index(new_index_name)
print(f"Created new index: {new_index_name} (dotproduct)")

Deleted old cosine index
Created new index: medical-hybrid-search (dotproduct)


In [16]:
print("Encoding sparse vectors and upserting...")

# Encode all sparse vectors at once (fast — BM25 is just word counting)

sparse_vecs = bm25.encode_documents(df['clinical_text'].astype(str).tolist())
                                                                             
# Build vector list in dict format (needed for sparse_values field)                                                                                                                                                                                                                                                                                                                                                             
vectors_to_upsert = []                                                                                                                                                                                                                                                                                                                                                                                                          
for i, (_, row) in enumerate(df.iterrows()):                                                                                                                                                                                                                                                                                                                                                                                    
    vectors_to_upsert.append({
        "id": row['unique_id'],
        "values": row['embedding'].tolist(),       # Dense (PubMedBERT)                                                                                                                                                                                                                                                                                                                                                         
        "sparse_values": sparse_vecs[i],           # Sparse (BM25)     
        "metadata": row['metadata']                                                                                                                                                                                                                                                                                                                                                                                             
    })                                                                                                                                                                                                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                                                                                                                                                                                                
batch_size = 100                                                                                                                                                                                                                                                                                                                                                                                                                
total_batches = (len(vectors_to_upsert) + batch_size - 1) // batch_size
                                                                       
for i in range(0, len(vectors_to_upsert), batch_size):                                                                                                                                                                                                                                                                                                                                                                          
    batch = vectors_to_upsert[i : i + batch_size]     
    hybrid_index.upsert(vectors=batch)                                                                                                                                                                                                                                                                                                                                                                                          
    print(f"Upserted batch {i // batch_size + 1}/{total_batches}")                                                                                                                                                                                                                                                                                                                                                              
                                                                                                                                                                                                                                                                                                                                                                                                                                
print(f"\nDone. {len(vectors_to_upsert)} hybrid vectors upserted.") 

Encoding sparse vectors and upserting...
Upserted batch 1/76
Upserted batch 2/76
Upserted batch 3/76
Upserted batch 4/76
Upserted batch 5/76
Upserted batch 6/76
Upserted batch 7/76
Upserted batch 8/76
Upserted batch 9/76
Upserted batch 10/76
Upserted batch 11/76
Upserted batch 12/76
Upserted batch 13/76
Upserted batch 14/76
Upserted batch 15/76
Upserted batch 16/76
Upserted batch 17/76
Upserted batch 18/76
Upserted batch 19/76
Upserted batch 20/76
Upserted batch 21/76
Upserted batch 22/76
Upserted batch 23/76
Upserted batch 24/76
Upserted batch 25/76
Upserted batch 26/76
Upserted batch 27/76
Upserted batch 28/76
Upserted batch 29/76
Upserted batch 30/76
Upserted batch 31/76
Upserted batch 32/76
Upserted batch 33/76
Upserted batch 34/76
Upserted batch 35/76
Upserted batch 36/76
Upserted batch 37/76
Upserted batch 38/76
Upserted batch 39/76
Upserted batch 40/76
Upserted batch 41/76
Upserted batch 42/76
Upserted batch 43/76
Upserted batch 44/76
Upserted batch 45/76
Upserted batch 46/76
Up

In [11]:
# Test with a highly specific medical query
query = "patient presenting with pulmonary nodules and respiratory distress"
query_embedding = model.encode(query, show_progress_bar=False).tolist()

query_results = index.query(
    vector=[query_embedding],
    top_k=5,
    include_metadata=True
)

score_threshold = 0.3

print(f"\n--- SEARCH RESULTS FOR: '{query}' ---\n")

for match in query_results['matches']:
    if match['score'] >= score_threshold:
        details = match.get('metadata', {})
        diagnosis = details.get('diagnosis', 'N/A')
        snippet = details.get('text_snippet', 'No snippet available')
        
        print(f"Score: {match['score']:.4f} | Diagnosis: {diagnosis}")
        print(f"Snippet: {snippet}")
        print("-" * 80)

UnauthorizedException: (401)
Reason: Unauthorized
HTTP response headers: HTTPHeaderDict({'Date': 'Sat, 14 Mar 2026 17:02:07 GMT', 'Content-Type': 'text/plain', 'Content-Length': '12', 'Connection': 'keep-alive', 'x-pinecone-auth-rejected-reason': 'Malformed domain', 'www-authenticate': 'Malformed domain', 'server': 'envoy'})
HTTP response body: Unauthorized
